# Module 18 — Distributed Task Queues: Interactive Verification

## What you will discover

Every cell runs the module's **real** implementation from `project_solution/`,
and every claim is asserted rather than printed. A notebook that prints a
plausible number teaches nothing.

1. That an idempotency key is derived from *content*, so two clients submitting
   the same work independently produce the same key.
2. That deduplication shows up in the **queue depth**, not in the returned
   job's id — and why that distinction matters.
3. That a permanently failing job is retried a bounded number of times and then
   dead-lettered, rather than retried forever.
4. That exponential backoff is capped, and what happens without the cap.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

`project_solution/` is added relative to this notebook's own location. Never
hard-code an absolute path — `tools/check_links.py` fails the build on them,
because a path with a username in it works on exactly one machine.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "project_solution"))

import distributed_pipeline as dp

print(f"loaded from: {Path(dp.__file__).parent.name}/")
print(f"JobStatus values: {[s.value for s in dp.JobStatus]}")
print(f"brokers available: {[n for n in dir(dp) if n.endswith('Broker')]}")

## 1. The idempotency key is derived from content, not from time

Two independent clients submitting the same logical work must produce the same
key, or deduplication cannot work at all. That means the key cannot depend on
anything incidental — not a timestamp, not a UUID, and **not dictionary
insertion order**.

In [ ]:
a = dp.derive_idempotency_key("send_email", {"to": "a@example.com", "template": 7})
b = dp.derive_idempotency_key("send_email", {"template": 7, "to": "a@example.com"})

print(f"key A: {a}")
print(f"key B: {b}")

assert a == b, "key must not depend on dict insertion order"

# sha256, truncated to 32 hex characters. 128 bits is far past any collision
# risk at queue scale, and truncating halves the bytes stored on every job.
assert len(a) == 32, f"expected a 32-char truncated sha256, got {len(a)}"

# A different payload must produce a different key, or everything collapses
# into one job.
c = dp.derive_idempotency_key("send_email", {"to": "a@example.com", "template": 8})
assert a != c, "different payloads must not collide"

# And a different task type with the same payload is different work.
d = dp.derive_idempotency_key("send_sms", {"to": "a@example.com", "template": 7})
assert a != d, "task type must participate in the key"

print("\nall four properties hold")

## 2. Predict before you run

A producer submits the **same** task twice, then a genuinely different one:

```python
producer.submit("send_email", {"id": 1})   # first
producer.submit("send_email", {"id": 1})   # identical
producer.submit("send_email", {"id": 2})   # different
```

**Write down your answers before running the next cell:**

1. What is `broker.queue_depth()` afterwards — 1, 2, or 3?
2. Does the second `submit` return a `Job` with the **same** `job_id` as the
   first?

Question 2 is the interesting one, and the obvious answer is wrong. Commit to
an answer before you run it.

In [ ]:
broker = dp.InMemoryBroker()
producer = dp.Producer(broker)

first = producer.submit("send_email", {"id": 1})
duplicate = producer.submit("send_email", {"id": 1})
different = producer.submit("send_email", {"id": 2})

print(f"queue_depth:          {broker.queue_depth()}")
print(f"first.job_id:         {first.job_id}")
print(f"duplicate.job_id:     {duplicate.job_id}")
print(f"same idempotency key: {first.idempotency_key == duplicate.idempotency_key}")

# The duplicate was NOT enqueued - two distinct units of work are queued.
assert broker.queue_depth() == 2, f"expected 2 queued, got {broker.queue_depth()}"

# But submit() still hands back a fresh Job object. Its job_id differs.
assert first.job_id != duplicate.job_id, "submit returns a new Job for a duplicate"

# The deduplication is visible in the shared idempotency key, not the job id.
assert first.idempotency_key == duplicate.idempotency_key
assert first.idempotency_key != different.idempotency_key

print("\nDedup is observable in queue_depth and the idempotency key.")
print("Asserting on job_id equality would fail - and would fail for the RIGHT")
print("reason, which is why it is worth knowing before you write that test.")

## 3. A permanently failing job is dead-lettered, not retried forever

The measurement that matters: a handler that *always* raises must be attempted
exactly `max_attempts` times and then moved to the dead-letter queue. Retrying
forever turns one poison message into an outage; dropping it silently loses
data.

In [ ]:
broker = dp.InMemoryBroker()
producer = dp.Producer(broker)
worker = dp.Worker(broker, name="worker-1")

attempts = {"n": 0}

def always_fails(payload):
    attempts["n"] += 1
    raise RuntimeError("downstream service is unreachable")

worker.register("send_email", always_fails)
producer.submit("send_email", {"id": 1}, max_attempts=3)

# Drain generously - more passes than could possibly be needed, so the test
# measures the retry policy rather than the loop count.
for _ in range(10):
    worker.run_once(count=5)

print(f"handler invocations: {attempts['n']}")
print(f"dead-lettered jobs:  {len(broker.dead_letter_jobs())}")
print(f"queue depth now:     {broker.queue_depth()}")

assert attempts["n"] == 3, f"expected exactly 3 attempts, got {attempts['n']}"
assert len(broker.dead_letter_jobs()) == 1, "the poison job must land in the DLQ"
assert broker.queue_depth() == 0, "and must not remain on the main queue"

dead = broker.dead_letter_jobs()[0]
print(f"\nfinal status: {dead.status.value}   attempts recorded: {dead.attempts}")
assert dead.status == dp.JobStatus.DEAD_LETTERED
assert dead.error is not None, "a dead-lettered job must carry why it died"

## 4. Backoff grows exponentially and is capped

Uncapped exponential backoff reaches absurd delays fast: doubling from 50 ms,
attempt 20 is over 14 hours. The cap is what makes the policy usable, and it is
worth seeing the two curves side by side.

In [ ]:
capped = [dp.exponential_backoff(i, base=0.05, cap=5.0) for i in range(14)]
uncapped = [0.05 * (2 ** i) for i in range(14)]

print(f"{'attempt':>7}  {'capped (s)':>11}  {'uncapped (s)':>13}")
for i, (c, u) in enumerate(zip(capped, uncapped, strict=True)):
    print(f"{i:>7}  {c:>11.3f}  {u:>13.2f}")

# Monotonically non-decreasing, and never above the cap.
assert all(b <= 5.0 for b in capped), "cap must be respected"
assert all(capped[i] <= capped[i + 1] for i in range(len(capped) - 1)), "must not decrease"
assert capped[0] == 0.05, "first retry waits one base interval"

# The cap actually binds within a realistic number of attempts.
assert capped[-1] == 5.0 and uncapped[-1] > 400, (
    "by attempt 13 the uncapped curve is already absurd"
)
print(f"\nuncapped attempt 20 would wait {0.05 * 2 ** 20 / 3600:.1f} hours")

## 5. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
# DELIBERATELY BROKEN - three expected values, two of them wrong. Fix in place.

expected_queue_depth = 3      # after two identical submits and one different
expected_attempts    = 5      # for a job with max_attempts=3 that always fails
expected_dlq_size    = 1      # poison jobs after the drain above

broker2 = dp.InMemoryBroker()
producer2 = dp.Producer(broker2)
producer2.submit("t", {"id": 1})
producer2.submit("t", {"id": 1})
producer2.submit("t", {"id": 2})

worker2 = dp.Worker(broker2, name="w2")
tries = {"n": 0}
def boom(payload):
    tries["n"] += 1
    raise RuntimeError("nope")
worker2.register("t", boom)
for _ in range(12):
    worker2.run_once(count=5)

assert broker2.queue_depth() + len(broker2.dead_letter_jobs()) >= 0  # sanity
assert expected_queue_depth == 2, f"queue_depth: expected {expected_queue_depth}, real answer differs"
assert expected_attempts == tries["n"] / 2, f"attempts per job: got {tries['n'] / 2}"
assert expected_dlq_size == len(broker2.dead_letter_jobs()) / 2

print("All three match. Now explain WHY each number is what it is.")

## Takeaways

1. **An idempotency key must be derived from content**, canonicalised so
   dictionary order cannot change it. A key that depends on a timestamp or a
   UUID deduplicates nothing.
2. **Deduplication is observable in `queue_depth` and the idempotency key, not
   in `job_id`.** `submit` returns a fresh `Job` for a duplicate. Asserting on
   `job_id` equality fails — for the right reason.
3. **Bounded retries plus a dead-letter queue** is the only combination that
   neither loses the message nor retries it forever.
4. **Backoff must be capped.** Uncapped doubling from 50 ms reaches 14 hours by
   attempt 20.
5. **A dead-lettered job must carry its error.** A DLQ of jobs with no reason
   attached is a list of things you cannot act on.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`project_solution/test_distributed_pipeline.py`](project_solution/test_distributed_pipeline.py) — the full test suite
- `debug_lab/` — planted defects to diagnose
- `starter/` — build it yourself; the shipped tests are the specification